In [0]:
dbutils.widgets.text("my_sql_host","35.226.28.96")
my_sql_host=dbutils.widgets.get("my_sql_host")

dbutils.widgets.text("my_sql_port","3306")
my_sql_port=dbutils.widgets.get("my_sql_port")

dbutils.widgets.text("my_sql_user","root")
my_sql_user=dbutils.widgets.get("my_sql_user")

dbutils.widgets.text("my_sql_pwd","Admin@1234")
my_sql_pwd=dbutils.widgets.get("my_sql_pwd")

dbutils.widgets.text("my_sql_db","GCPMigrationmMeta")
my_sql_db=dbutils.widgets.get("my_sql_db")

dbutils.widgets.text("sak","/Volumes/workspace/default/gcptodbmigration/sk/datamigrationproject-494118-b1bc74441af6.json")
sak=dbutils.widgets.get("sak")

dbutils.widgets.text("hive_db","bronze")
hive_db=dbutils.widgets.get("hive_db")

In [0]:
%pip install -q mysql-connector-python google-cloud-bigquery google-cloud-storage

In [0]:
import os,re,shutil,json
import mysql.connector as sql
from contextlib import contextmanager
from urllib.parse import urlparse
from google.cloud import storage
from pyspark.sql.functions import *
from pyspark.sql.types import *

MySql={
"host":my_sql_host,
"port":my_sql_port,
"user":my_sql_user,
"pwd":my_sql_pwd,
"db":my_sql_db
}

assert os.path.exists(sak), f"GCP key not found at {sak}"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"]=sak

In [0]:
@contextmanager
def my_sql_conn():
    conn=sql.connect(
        host=MySql["host"],
        port=int(MySql["port"]),
        user=MySql["user"],
        password=MySql["pwd"],
        database=MySql["db"]
    )
    try:
        yield conn
    finally:
        conn.close()

In [0]:
def fetch_eligible_rows():
    with my_sql_conn() as conn:
        cur=conn.cursor(dictionary=True)
        cur.execute("""select table_name, gcs_path, target_path
                    from config_table
                    where active_flag=1
                    and load_falg=1
                    and bq_to_gcs_status="COMPLETED"
                    and gcs_to_bronze_status in ("NOT_STARTED","FAILED")
                    order by table_name""")
        rows=cur.fetchall()
        cur.close()
    return rows

In [0]:
def set_bronze_status(table_name,status,error=None):
    with my_sql_conn() as conn:
        cur=conn.cursor()
        if status=="IN_PROGRESS":
            cur.execute("""update config_table
                        set gcs_to_bronze_status="IN_PROGRESS",
                        last_run_ts=NOW(), error_message=NULL
                        where table_name=%s""",(table_name,))
        elif status=="COMPLETED":
            cur.execute("""update config_table
                        set gcs_to_bronze_status="COMPLETED",
                        last_run_ts=NOW(), error_message=NULL
                        where table_name=%s""",(table_name,))
        else:
            cur.execute("""update config_table
                        set gcs_to_bronze_status="FAILED",
                        last_run_ts=NOW(), error_message=%s
                        where table_name=%s""",(str(err)[:2000] if err else "FAILED",table_name))
        conn.commit()
        cur.close()

In [0]:
def reset_load_flag(table_name):
    with my_sql_conn() as conn:
        cur=conn.cursor()
        cur.execute("""update config_table
                    set load_falg=0
                    where table_name=%s""",(table_name,))
        conn.commit()
        cur.close()

In [0]:
def _gcs_client():
    return storage.Client()